# Deep Agents and Jev
### When and how to use system one models with deep agents

Jev is not a traditional LLM, it _does not_ generate text. It’s what the TypeSafe AI team calls a System One model:

📖 System One models are a class of AI models built to make fast, structured decisions that software can use directly. A System One model evaluates a state and returns typed answers and probabilities.

To invoke a Jev model, you send it a state (the context) and questions about that state. Here’s an example of classifying a support ticket request:

<pre style="background:#f6f8fa; color:#1f2328; border:1px solid #d0d7de; border-radius:6px; padding:12px 16px; line-height:1.45; overflow-x:auto;"><code><span style="color:#57606a">{</span>
  <span style="color:#0550ae">"model"</span><span style="color:#57606a">:</span> <span style="color:#0a3069">"jev-latest"</span><span style="color:#57606a">,</span>
  <span style="color:#0550ae">"state"</span><span style="color:#57606a">:</span> <span style="color:#0a3069">"Hi, I've been trying to connect my Stripe account for 3 days and it keeps failing. I'm losing sales. Please help ASAP."</span><span style="color:#57606a">,</span>
  <span style="color:#0550ae">"questions"</span><span style="color:#57606a">:</span> <span style="color:#57606a">{</span>
    <span style="color:#0550ae">"is_urgent"</span><span style="color:#57606a">:</span> <span style="color:#57606a">{</span>
      <span style="color:#0550ae">"type"</span><span style="color:#57606a">:</span> <span style="color:#0a3069">"noul"</span><span style="color:#57606a">,</span>
      <span style="color:#0550ae">"instructions"</span><span style="color:#57606a">:</span> <span style="color:#0a3069">"The message conveys urgency or time-sensitivity"</span>
    <span style="color:#57606a">}</span>
  <span style="color:#57606a">}</span>
<span style="color:#57606a">}</span></code></pre>
What you get back is **typed response** with a probability:

<pre style="background:#f6f8fa; color:#1f2328; border:1px solid #d0d7de; border-radius:6px; padding:12px 16px; line-height:1.45; overflow-x:auto;"><code><span style="color:#57606a">{</span>
  <span style="color:#0550ae">"is_urgent"</span><span style="color:#57606a">:</span> <span style="color:#57606a">{</span>
    <span style="color:#0550ae">"type"</span><span style="color:#57606a">:</span> <span style="color:#0a3069">"noul"</span><span style="color:#57606a">,</span>
    <span style="color:#0550ae">"noul"</span><span style="color:#57606a">:</span> <span style="color:#953800">0.999</span>
  <span style="color:#57606a">}</span>
<span style="color:#57606a">}</span></code></pre>

<img src="images/jev-response-types.png" alt="Jev's three response types: Noul (is this true?) returns a probability, Choice (pick one option) returns a distribution over options plus confidence, and Score (place on a scale) returns a score plus confidence." style="display:block; margin:0 auto; max-width:100%; width:960px; border:1px solid #d0d7de; border-radius:6px;" />

# Three ways to use Jev in a deep agent

Every example below works the support queue at **Acme**, a small SaaS company. We'll use Jev three ways:

1. **Classifying**: answer typed questions about a ticket, and give that ability to a deep agent as a tool.
2. **Model routing**: let Jev pick the cheapest model that can handle each request.
3. **Judging**: grade the agent's replies against a policy, fast enough to check every one.

The integration lives in <code style="font-size:15px; color:#2b8a3e;">langchain-typesafe</code> and reads your key from <code style="font-size:15px; color:#2b8a3e;">TYPESAFE_API_KEY</code>.

In [10]:
import warnings

from dotenv import load_dotenv
load_dotenv(override=True)

from deepagents import create_deep_agent
from langchain_core._api import LangChainBetaWarning
from langchain_typesafe import Choice, Noul, Score, TypeSafeClassifier
from util import print_exchange, print_tool_trace

warnings.filterwarnings("ignore", category=LangChainBetaWarning)

fast_model = "anthropic:claude-haiku-4-5-20251001"
powerful_model = "anthropic:claude-sonnet-5-5"

classifier = TypeSafeClassifier()

# 1. Classifying

<code style="font-size:15px; color:#2b8a3e;">TypeSafeClassifier</code> is a LangChain runnable. Pass it a **state** and some **questions**, and it returns typed answers. Here's the example from the blog post: one <code style="font-size:15px; color:#2b8a3e;">Noul</code> (yes/no) question about a single message.

In [11]:
response = classifier.invoke({
    "state": "Our deploy failed and customers are seeing 500s on checkout.",
    "questions": {
        "urgent": Noul(instructions="Does this need attention right now?"),
    },
})

print(response.model_dump_json(indent=2))

{
  "model": "jev-1.13.0",
  "answers": {
    "urgent": {
      "type": "noul",
      "noul": 0.97
    }
  },
  "usage": {
    "input_tokens": 288,
    "output_tokens": 20
  },
  "request_id": "req_01a0fdff6aac75549c56772d2d4604d5"
}


## Ask several questions at once

Questions in one request are answered in parallel, so a second or third question adds little latency or cost. Here a <code style="font-size:15px; color:#2b8a3e;">Noul</code>, a <code style="font-size:15px; color:#2b8a3e;">Choice</code> and a <code style="font-size:15px; color:#2b8a3e;">Score</code> share the same ticket.

In [12]:
ticket_questions = {
    "urgent": Noul(instructions="Does this need attention right now?"),
    "team": Choice(
        instructions="Which team should handle this ticket?",
        criteria={
            "billing": "Payments, invoices, refunds, or subscription changes.",
            "technical": "Bugs, outages, errors, or integration failures.",
            "account": "Login, password, or user management.",
        },
    ),
    "frustration": Score(
        instructions="How frustrated is the customer?",
        criteria=["Calm", "Concerned", "Angry"],
    ),
}

response = classifier.invoke({
    "state": "Third time asking: I was double charged for September. Fix it today.",
    "questions": ticket_questions,
})

print("urgent:     ", response.nouls["urgent"].noul)
print("team:       ", response.choices["team"].choice, response.choices["team"].probabilities)
print("frustration:", response.scores["frustration"].score, "on a 0-2 scale")

urgent:      0.9
team:        billing {'account': 0.0, 'technical': 0.0, 'billing': 1.0}
frustration: 2.0 on a 0-2 scale


## Give the classifier to a deep agent

Wrap the call in a tool, and the agent can triage tickets whenever a task calls for it. Jev does the judging in milliseconds and returns numbers the agent can sort and reason over. The agent keeps the parts an LLM is good at: deciding what to do and explaining it.

In [13]:
def triage_ticket(ticket: str) -> dict:
    """Classify a support ticket: which team owns it, and how likely it is to be urgent (0-1)."""
    response = classifier.invoke({"state": ticket, "questions": ticket_questions})
    return {
        "team": response.choices["team"].choice,
        "urgency": response.nouls["urgent"].noul,
        "frustration": response.scores["frustration"].score,
    }


support_lead = create_deep_agent(
    model=fast_model,
    tools=[triage_ticket],
    system_prompt="You are the support lead at Acme, a small SaaS company.",
)

inbox = """1. Can you add my colleague sam@example.com to our workspace?
2. Our deploy failed and customers are seeing 500s on checkout.
3. I was double charged for September, please fix when you get a chance."""

result = support_lead.invoke({
    "messages": [{"role": "user", "content": f"Here's this morning's inbox. What should we work on first?\n\n{inbox}"}],
})
print_tool_trace(result)
print_exchange(result)

### Tool trace

- → 🔧 **triage_ticket**({'ticket': 'Can you add my colleague sam@example.com to our workspace?'})
- → 🔧 **triage_ticket**({'ticket': 'Our deploy failed and customers are seeing 500s on checkout.'})
- → 🔧 **triage_ticket**({'ticket': 'I was double charged for September, please fix when you get a chance.'})
- ← 📥 `triage_ticket` → {"team": "account", "urgency": 0.4, "frustration": 0.0}
- ← 📥 `triage_ticket` → {"team": "technical", "urgency": 0.98, "frustration": 1.21}
- ← 📥 `triage_ticket` → {"team": "billing", "urgency": 0.57, "frustration": 0.45}

### Human

Here's this morning's inbox. What should we work on first?

1. Can you add my colleague sam@example.com to our workspace?
2. Our deploy failed and customers are seeing 500s on checkout.
3. I was double charged for September, please fix when you get a chance.

### Ai

**Priority Order (work on these in this order):**

1. **🔴 URGENT - Deploy/Checkout Issue (Technical Team)**
   - Urgency: 0.98/1.0
   - Customers are seeing 500 errors on checkout - this is actively costing revenue and damaging customer trust. This needs immediate attention.

2. **🟡 MEDIUM - Billing Issue (Billing Team)**
   - Urgency: 0.57/1.0
   - Double charge for September needs investigation. The customer seems patient ("when you get a chance"), but this affects customer retention and trust.

3. **🟢 LOW - Workspace Access (Account Team)**
   - Urgency: 0.4/1.0
   - Adding a colleague to the workspace is a routine request with no time pressure.

**Recommendation:** Get the technical team on the checkout 500 errors immediately. Once that's stabilized, have billing look into the double charge. The workspace addition can wait until after these are resolved.

# 2. Model routing

Most support questions are quick lookups. A few need careful reasoning. <code style="font-size:15px; color:#2b8a3e;">ModelRouterMiddleware</code> asks Jev a <code style="font-size:15px; color:#2b8a3e;">Choice</code> question about the latest user message and sends every model call in the run to the model it picks.

- The routing runs once per invocation, before the agent starts.
- The full answer, including probabilities and confidence, is saved in agent state as <code style="font-size:15px; color:#2b8a3e;">model_route</code>.
- The choices and criteria below come from the blog post.

In [14]:
from langchain_typesafe.experimental.middleware import ModelChoice, ModelRouterMiddleware

router = ModelRouterMiddleware(
    choices={
        "fast": ModelChoice(model=fast_model, criteria="Direct lookups, extraction, and localized changes."),
        "powerful": ModelChoice(model=powerful_model, criteria="Architecture and high-stakes decisions."),
    },
    instructions="Choose the least costly model that can complete the task.",
)

routed_agent = create_deep_agent(
    model=fast_model,
    middleware=[router],
    system_prompt="You are a support engineer at Acme, a small SaaS company. Be concise.",
)

Let's send one easy question and one hard one, then check which model Jev picked and which model actually answered.

In [ ]:
def ask(question: str) -> None:
    result = routed_agent.invoke({"messages": [{"role": "user", "content": question}]})
    route = result["model_route"]
    answered_by = {m.response_metadata.get("model_name") for m in result["messages"] if m.type == "ai"}
    print(f"route: {route.choice}  probabilities: {route.probabilities}  answered by: {answered_by}")
    print_exchange(result)


ask("What HTTP status code means 'Too Many Requests'?")

In [ ]:
ask("Our checkout service drops orders when Stripe webhooks arrive out of order. "
    "Design a fix we can roll out without downtime.")

<img src="images/llm-vs-jev.png" alt="An LLM reads a context window and writes free text. Jev reads a state plus questions and returns a set of probabilities." style="max-width:100%; width:960px; border:1px solid #d0d7de; border-radius:6px;" />

# 3. Jev as a judge

An LLM judge writes an opinion that you then have to parse, and it's often slower and pricier than the model it's grading. Jev returns calibrated numbers in about 100 ms, so you can check **every** reply rather than a sample.

The state doesn't have to be a single string. Here it's a JSON object holding the refund policy, the customer's ticket and the agent's reply, so Jev can compare them.

In [ ]:
refund_policy = """Acme refund policy:
- Monthly plans: full refund within 14 days of a charge, none after.
- Annual plans: full refund within 30 days; after that, a prorated refund for unused months.
- Usage overages are never refunded."""

support_agent = create_deep_agent(
    model=fast_model,
    system_prompt=f"You are a support agent at Acme. Reply to the customer in 2-3 sentences.\n\n{refund_policy}",
)

judge_questions = {
    "follows_policy": Noul(instructions="Is everything the reply promises allowed by the policy?"),
    "answers_ticket": Noul(instructions="Does the reply directly address the customer's request?"),
    "tone": Score(
        instructions="How professional and empathetic is the reply's tone?",
        criteria=["Rude or dismissive", "Neutral", "Warm and professional"],
    ),
}


def judge(ticket: str, reply: str) -> None:
    response = classifier.invoke({
        "state": {"policy": refund_policy, "ticket": ticket, "reply": reply},
        "questions": judge_questions,
    })
    print(
        f"follows_policy: {response.nouls['follows_policy'].noul:.2f}  "
        f"answers_ticket: {response.nouls['answers_ticket'].noul:.2f}  "
        f"tone: {response.scores['tone'].score:.2f} / 2"
    )

Let's grade the agent on three refund requests, one for each rule in the policy.

In [ ]:
tickets = [
    "I was charged for my monthly plan 5 days ago but we've stopped using Acme. Can I get my money back?",
    "We're 4 months into our annual plan and shutting down the project. What can you refund?",
    "We went way over our API quota last month and got a $400 overage bill. Please refund it.",
]

for ticket in tickets:
    reply = support_agent.invoke({"messages": [{"role": "user", "content": ticket}]})["messages"][-1].text
    print(f"TICKET: {ticket}\nREPLY:  {reply}")
    judge(ticket, reply)
    print()

A judge only helps if it catches bad replies, so let's grade two we wrote by hand. The first is friendly but breaks the policy, and the second follows the policy but is rude.

In [ ]:
overage_ticket = tickets[2]

judge(overage_ticket, "So sorry about that! I've refunded the full $400 overage; it should land in 3-5 days.")
judge(overage_ticket, "Overages aren't refundable. Read the policy next time.")